**COMPUTATIONAL TIME: BRAIN STRUCTURES**

In [1]:
from memory_profiler import memory_usage

# Function to just return nothing, for baseline memory snapshot
def baseline_memory():
    return None

# Peak memory at the very beginning
baseline_peak = max(memory_usage(baseline_memory, interval=0.01, max_iterations=10))
print(f"Baseline memory before loading any model: {baseline_peak:.2f} MB")


Baseline memory before loading any model: 74.01 MB


In [2]:
from monai.networks.nets.autoencoderkl import Encoder
from monai.networks.nets.autoencoderkl import AutoencoderKL
from model.contrastive_model import ContrastiveModel
import matplotlib.pyplot as plt
import torch
import os
import psutil
import time
import gc
from tqdm import tqdm

gc.collect()  # Clean up any unreachable objects
%load_ext memory_profiler


# Limit PyTorch to 1 thread
torch.set_num_threads(1)

# Optional: also restrict OMP and MKL
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# AUTOENCODER
device = "cpu" # torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
config_vae = {
    "random_state": 1234,
    "data_path": "/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/original/",
    "save_path": "/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/region_brain/",
    "metadata_file": "/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/combined_metadata.csv", 
    "labels_path": "data/labels.csv",
    "bb_path": "data/bounding_boxes.csv",
    "ckpt_path": "/cephyr/users/felixnie/Alvis/logs/20250725_110120/checkpoint-epoch-12.pth",
    "use_old_state_dict": False,
    "n_structs": -1,
    "vae_params": {
        "spatial_dims": 3,
        "in_channels": 1,
        "out_channels": 1,
        "latent_channels": 8,
        "channels": [64, 128, 128, 128],
        "num_res_blocks": 2,
        "norm_num_groups": 32,
        "norm_eps": 1e-6,
        "attention_levels": [False, False, False, False],
        "with_decoder_nonlocal_attn": False,
        "with_encoder_nonlocal_attn": False,
    },
    "batch_size": 64,
}

# CL PROJECTOR
config_cl = {
    "data_path": "/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/",
    "output_dir": "data/results/region_brain/eval_cl16/",
    "dataset_index_file_name": "region_brain/dataset_index.csv",
    "metadata_file_name": "combined_metadata.csv",
    "labels_path": "data/labels.csv",
    "bb_path": "data/bounding_boxes.csv",
    "proj_params": {
        "input_shape": [8, 8, 8, 8],
        "projector_dims": [128],
        "final_dim": 16
    },
    "encoder_params": {
        "spatial_dims": 3,
        "in_channels": 8,
        "channels": [64, 128, 256],
        "out_channels": 256,
        "num_res_blocks": [2, 2, 2],
        "norm_num_groups": 8,
        "norm_eps": 1e-5,
        "attention_levels": [False, False, False],
        "with_nonlocal_attn": False,
        "include_fc": False,
    },
}
config_cl.update({
    "resume_path": "/cephyr/users/felixnie/Alvis/logs/20250804_001927/checkpoint-epoch-495.pth",
    "n_structs": -1,
    "batch_size": 256
})



In [4]:
# Peak memory at the very beginning
baseline_peak = max(memory_usage(baseline_memory, interval=0.01, max_iterations=10))
print(f"Baseline memory before loading any model: {baseline_peak:.2f} MB")

Baseline memory before loading any model: 752.03 MB


In [5]:
import torch
import time
import gc
from memory_profiler import memory_usage

# ----------------------------
# Function to measure memory usage during a callable
# ----------------------------
def measure_memory(func, *args, **kwargs):
    """
    Runs func(*args, **kwargs) and returns:
      - result: function output
      - peak_mem: peak memory during execution in MB
    """
    peak_mem = memory_usage((func, args, kwargs), interval=0.01, max_iterations=1)
    return peak_mem, func(*args, **kwargs)

# ----------------------------
# VAE loading
# ----------------------------
def load_vae_model(config, device, ckpt_path=None, use_old_state_dict=False):
    vae_params = config["vae_params"]
    autoencoder = AutoencoderKL(**vae_params).to(device)

    if ckpt_path:
        checkpoint = torch.load(ckpt_path, map_location=device)
        ckpt_key = config.get("ckpt_key", "autoencoder_state_dict")
        if use_old_state_dict:
            autoencoder.load_old_state_dict(checkpoint[ckpt_key])
            print("Loaded weights using load_old_state_dict().")
        else:
            autoencoder.load_state_dict(checkpoint[ckpt_key])
            print("Loaded weights using load_state_dict().")
    return autoencoder

# ----------------------------
# Measure memory for VAE load
# ----------------------------
gc.collect()
time_before = time.time()
mem_peak_vae, autoencoder = measure_memory(load_vae_model,
                                           config=config_vae,
                                           device=device,
                                           ckpt_path=config_vae.get("ckpt_path"),
                                           use_old_state_dict=config_vae.get("use_old_state_dict"))
autoencoder.eval()
time_after = time.time()
print(f"VAE Load Time: {time_after - time_before:.2f} s")
print(f"Peak memory during VAE load: {max(mem_peak_vae):.2f} MB")

# ----------------------------
# Contrastive Learning (CL) projector
# ----------------------------
def create_encoder(config, device):
    encoder_params = config["encoder_params"]
    return Encoder(**encoder_params).to(device)

def build_cl_model(config, device):
    encoder = create_encoder(config, device)
    model = ContrastiveModel(
        encoder=encoder,
        input_shape=config["proj_params"]["input_shape"],
        projector_dims=config["proj_params"]["projector_dims"],
        final_dim=config["proj_params"]["final_dim"],
        device=device
    ).to(device)
    checkpoint = torch.load(config["resume_path"], map_location=device)
    model.load_state_dict(checkpoint['state_dict'])
    return model

gc.collect()
time_before_cl = time.time()
mem_peak_cl, model = measure_memory(build_cl_model, config_cl, device)
model.eval()
time_after_cl = time.time()

print(f"CL Model Load Time: {time_after_cl - time_before_cl:.2f} s")
print(f"Peak memory during CL load: {max(mem_peak_cl):.2f} MB")


Loaded weights using load_state_dict().
Loaded weights using load_state_dict().
VAE Load Time: 0.60 s
Peak memory during VAE load: 999.29 MB
CL Model Load Time: 0.42 s
Peak memory during CL load: 1012.87 MB


In [6]:
from preprocessing.load_dataset import SubCorBatDataset
import pandas as pd
import os
import numpy as np

### Input data
# Path to dataset
DATA_PATH = "/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/"
load_ds_path = DATA_PATH + "batched_adni/"
# Files to load/save extension
extension = ".npz"
# Pretrained weights for the VAE
ckpt_vae_path = ""#"./data/pretrained_models/autoencoder_puglisi.pth"
# Pretrained weights for the Discriminator
ckpt_dis_path = ""#"./data/pretrained_models/ckpt_dis_.pth"
# Preparing image for using as input of the VAE
target_shape = [1, 160, 224, 160] # Desired shape: [1, 160, 224, 160]

# Load metadata
index_ds = pd.read_csv(os.path.join(DATA_PATH,"original/dataset_index.csv"))
clinical_ds = pd.read_csv(os.path.join(DATA_PATH,"combined_metadata.csv"))
metadata = pd.merge(index_ds, clinical_ds, on="GUID", how="inner") # Merge on the 'GUID' column
print(f"METADATA: Original rows: {len(metadata)}")

# First, ensure empty strings are treated as NaN
metadata['subject'].replace('', pd.NA, inplace=True)

# Then drop rows where subject is NaN
metadata = metadata.dropna(subset=['subject'])

# Optional: reset index
metadata = metadata.reset_index(drop=True)
print(f"METADATA: Remaining rows: {len(metadata)}") # Check result

# Load labels

# Training configuration
batch_files = sorted(metadata["batch_file"].unique())

# Load labels and bounding boxes
labels_df = pd.read_csv("data/labels.csv")
bb_df = pd.read_csv("data/bounding_boxes.csv")
labels_bb_df = pd.merge(labels_df, bb_df, on="LabelName", how="inner") # Merge on the 'GUID' column

batch_files = batch_files[-4:-3] # To only load ABLI
print(batch_files)

from preprocessing.load_dataset import SingleStructDataset
dataset = SingleStructDataset(metadata, batch_files, labels_bb_df, target_struct_name="Left-Hippocampus")

guids = np.array(dataset.guids)

sample = dataset[0].copy()
struct = sample["image"].unsqueeze(0).type(torch.float32).to(device)

del dataset
gc.collect()

METADATA: Original rows: 26685
METADATA: Remaining rows: 26685
['/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/original/batched_aibl/batch_0002.npz']


0

In [7]:
times_total = []
times_encode = []
times_project = []

n_iter = 10

for _ in tqdm(range(n_iter)):
    

    tic = time.time()
    with torch.no_grad():
        z_mu_batch, _ = autoencoder.encode(struct)
    toc = time.time()
    with torch.no_grad():
        proj_embs = model(z_mu_batch).cpu().numpy()
    tuc = time.time()

    times_total.append(tuc - tic)
    times_encode.append(toc - tic)
    times_project.append(tuc - toc)


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:31<00:00,  3.19s/it]


In [8]:
from memory_profiler import memory_usage

mem_total = []
mem_encode = []
mem_project = []



for _ in tqdm(range(n_iter)):
    # Encode memory
    def encode_step():
        with torch.no_grad():
            z_mu_batch, _ = autoencoder.encode(struct)
        return z_mu_batch

    mem_encode_peak = max(memory_usage(encode_step, interval=0.01))
    z_mu_batch = encode_step()  # actually run

    # Project memory
    def project_step():
        with torch.no_grad():
            proj_embs = model(z_mu_batch).cpu().numpy()
        return proj_embs

    mem_project_peak = max(memory_usage(project_step, interval=0.01))
    proj_embs = project_step()

    # Total memory for this iteration
    mem_total_peak = max(mem_encode_peak, mem_project_peak)

    mem_encode.append(mem_encode_peak)
    mem_project.append(mem_project_peak)
    mem_total.append(mem_total_peak)


  0%|                                                                                                                                                                                                      | 0/10 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [01:06<00:00,  6.70s/it]


In [9]:

# Convert to numpy
times_total = np.array(times_total)
times_encode = np.array(times_encode)
times_project = np.array(times_project)

mem_total = np.array(mem_total)
mem_encode = np.array(mem_encode)
mem_project = np.array(mem_project)

# Print results
print(f"\n=== Runtime Summary ({n_iter} iterations) ===")
print(f"Total   : {times_total.mean():.6f} ± {times_total.std():.6f} sec")
print(f"Encode  : {times_encode.mean():.6f} ± {times_encode.std():.6f} sec")
print(f"Project : {times_project.mean():.6f} ± {times_project.std():.6f} sec")

print(f"\n=== Memory Summary ({n_iter} iterations, peak MB) ===")
print(f"Total   : {mem_total.mean():.2f} ± {mem_total.std():.2f} MB")
print(f"Encode  : {mem_encode.mean():.2f} ± {mem_encode.std():.2f} MB")
print(f"Project : {mem_project.mean():.2f} ± {mem_project.std():.2f} MB")
print("Results limited to only CPU 1 core.")
print()

# CPU info
import platform
import cpuinfo

print("CPU threads PyTorch will use:", torch.get_num_threads())
print("Available devices:", torch.get_num_interop_threads())  # inter-op parallelism

print("Platform:", platform.system(), platform.release())
print("Machine:", platform.machine())

info = cpuinfo.get_cpu_info()
print("CPU brand:", info["brand_raw"])
print("Arch:", info["arch"])
print("Cores (logical):", info["count"])
print("Frequency (Hz):", info["hz_advertised_friendly"])


=== Runtime Summary (10 iterations) ===
Total   : 3.192057 ± 0.063331 sec
Encode  : 3.167901 ± 0.062402 sec
Project : 0.024155 ± 0.002626 sec

=== Memory Summary (10 iterations, peak MB) ===
Total   : 1444.08 ± 0.23 MB
Encode  : 1444.08 ± 0.23 MB
Project : 1188.62 ± 0.00 MB
Results limited to only CPU 1 core.

CPU threads PyTorch will use: 1
Available devices: 32
Platform: Linux 4.18.0-553.47.1.el8_10.x86_64
Machine: x86_64
CPU brand: Intel(R) Xeon(R) Gold 6226R CPU @ 2.90GHz
Arch: X86_64
Cores (logical): 32
Frequency (Hz): 2.9000 GHz


In [15]:
from tabulate import tabulate
import numpy as np

time_vae_load = time_after - time_before
time_cl_load = time_after_cl - time_after
time_total_load = time_after_cl - time_before

# Build table
summary = [
    ["Load VAE encoder", f"{time_vae_load:.2f}"],
    ["Load CL projector", f"{time_cl_load:.2f}"],
    ["Total load", f"{time_total_load:.2f}"],
    ["Run VAE Encode", f"{times_encode.mean():.2f} ± {times_encode.std():.2f}"],
    ["Run CL Project", f"{times_project.mean():.2f} ± {times_project.std():.2f}"],
    ["Total run", f"{times_total.mean():.2f} ± {times_total.std():.2f}"]
]

# Print table
print(tabulate(summary, headers=["Step", "Time (s)"], tablefmt="grid"))




+-------------------+-------------+
| Step              | Time (s)    |
+===================+=============+
| Load VAE encoder  | 0.60        |
+-------------------+-------------+
| Load CL projector | 0.62        |
+-------------------+-------------+
| Total load        | 1.22        |
+-------------------+-------------+
| Run VAE Encode    | 3.17 ± 0.06 |
+-------------------+-------------+
| Run CL Project    | 0.02 ± 0.00 |
+-------------------+-------------+
| Total run         | 3.19 ± 0.06 |
+-------------------+-------------+
